# A. Острова рекомендаций

Решение пяти вопросов по таблицам `items_A.csv` и `also_viewed_A.csv`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("task_vseros_0")
items = pd.read_csv(data_dir / "items_A.csv")
also_viewed = pd.read_csv(data_dir / "also_viewed_A.csv")


## A1

Считаем телефоны с рейтингом не ниже 4.5, которые есть в наличии.


In [ ]:
answer1 = items[(items["category"] == "phones") & (items["rating"] >= 4.5) & (items["in_stock"] == 1)].shape[0]
print("A1:", answer1)


A1: 22


## A2

Для каждого бренда считаем среднюю цену ноутбуков и выбираем максимальную.


In [ ]:
laptop_mean_prices = (items[items["category"] == "laptops"].groupby("brand")["price"].mean().sort_values(ascending=False))
answer2 = laptop_mean_prices.index[0]

print(laptop_mean_prices)
print("A2:", answer2)


brand
Gamma       165387.500000
WorkPro     133715.700000
SoundX      133296.500000
Delta       132369.454545
ZenTech     132348.181818
Alpha       125091.428571
Beta        122253.666667
CaseCo      114983.416667
DeepAI      103704.285714
ReadMore     86510.000000
Name: price, dtype: float64
A2: Gamma


## A3

Размечаем каждый товар как `premium`, `standard` или `budget`, затем считаем доступные товары `premium`.


In [ ]:
items["segment"] = np.select(
    [
        (items["rating"] >= 4.5) & (items["price"] >= 50_000),
        (items["rating"] >= 4.0) & (items["price"] < 50_000),
    ],
    ["premium", "standard"],
    default="budget",
)

answer3 = items[(items["segment"] == "premium") & (items["in_stock"] == 1)].shape[0]
print("A3:", answer3)


A3: 25


## A4

Оставляем разные `item_to`, добавляем категории товаров и считаем их количество. Затем добавляем отсутствующие категории с нулём и сохраняем результат в `answer4.csv`.


In [5]:
answer4 = (
    also_viewed[["item_to"]]
    .drop_duplicates()
    .merge(items[["item_id", "category"]], left_on="item_to", right_on="item_id")
    .groupby("category")
    .size()
    .reindex(sorted(items["category"].unique()), fill_value=0)
    .rename("cnt")
    .reset_index()
)

answer4.to_csv(data_dir / "answer4.csv", index=False)
print(answer4.to_string(index=False))


   category  cnt
accessories   53
      books   14
       home   13
    laptops   16
     phones   89
       toys   12


## A5

Связи считаются неориентированными. Строим список соседей и с помощью поиска в глубину находим компоненты связности. Компоненту учитываем, если в ней есть и `phones`, и `accessories`.


In [6]:
category = dict(zip(items["item_id"], items["category"]))
neighbors = {item_id: set() for item_id in items["item_id"]}

for item_from, item_to in also_viewed.itertuples(index=False, name=None):
    neighbors[item_from].add(item_to)
    neighbors[item_to].add(item_from)

visited = set()
answer5 = 0

for start in neighbors:
    if start in visited:
        continue

    stack = [start]
    island_categories = set()

    while stack:
        item_id = stack.pop()
        if item_id in visited:
            continue
        visited.add(item_id)
        island_categories.add(category[item_id])
        stack.extend(neighbors[item_id] - visited)

    if "phones" in island_categories and "accessories" in island_categories:
        answer5 += 1

print("A5:", answer5)


A5: 11


## Ответы

- **A1:** `22`
- **A2:** `Gamma`
- **A3:** `25`
- **A4:** результат сохранён в `answer4.csv`
- **A5:** `11`
